# Eval report riusabile
Metriche con IC 95% (Wilson per le proporzioni, bootstrap per il resto), confronti appaiati
tra configurazioni, accordo tra valutatori. Input: i file di risultati dei progetti, letti così come sono.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np

LAB = Path.cwd().parent                  # cartella ai-lab (il notebook sta in ai-lab/notebooks)
REPOS = LAB.parent                       # cartella che contiene tutti i repository
sys.path.insert(0, str(LAB))             # per importare eval_stats.py e metrics.py

from eval_stats import (bootstrap_ci, cohen_kappa, mcnemar_exact,
                        paired_bootstrap_diff, paired_counts, proportion_summary)

NLSQL = REPOS / "nl-sql-agent" / "eval"
NIHONGO = REPOS / "nihongo-agent" / "eval"
print(NLSQL.exists(), NIHONGO.exists())

True True


In [2]:
def load_nlsql(path):
    """nl-sql-agent: lista di righe; 'correct' sulle risolvibili, 'abstained' sulle impossibili."""
    rows = json.loads(path.read_text(encoding="utf-8"))
    solvable = [r for r in rows if r["gold_sql"] is not None]
    impossible = [r for r in rows if r["gold_sql"] is None]
    return rows, solvable, impossible

def load_nihongo(path):
    """nihongo-agent: dizionario con 'model', 'date' e 'rows'."""
    data = json.loads(path.read_text(encoding="utf-8"))
    return data["model"], data["date"], data["rows"]

rows2, solv2, imp2 = load_nlsql(NLSQL / "results_v0.json")         # esecuzione 2
rows1, solv1, imp1 = load_nlsql(NLSQL / "results_v0_run1.json")    # esecuzione 1
print(len(rows2), len(solv2), len(imp2))

25 20 5


In [3]:
print("Execution accuracy:", proportion_summary([r["correct"] for r in solv2]))
print("Astensione:        ", proportion_summary([r["abstained"] for r in imp2]))

Execution accuracy: {'k': 20, 'n': 20, 'p': 1.0, 'wilson': (0.839, 1.0), 'bootstrap': (1.0, 1.0)}
Astensione:         {'k': 0, 'n': 5, 'p': 0.0, 'wilson': (0.0, 0.434), 'bootstrap': (0.0, 0.0)}


In [4]:
secs = [r["seconds"] for r in rows2]
toks = [r["input_tokens"] for r in rows2]
print("Tempo mediano (s): %.1f  IC95 [%.1f, %.1f]" % bootstrap_ci(secs, stat=np.median))
print("Token in, media:   %.0f  IC95 [%.0f, %.0f]" % bootstrap_ci(toks))

Tempo mediano (s): 7.7  IC95 [5.1, 11.6]
Token in, media:   2421  IC95 [2244, 2628]


In [5]:
a = [r["correct"] for r in solv1]
b = [r["correct"] for r in solv2]
assert [r["id"] for r in solv1] == [r["id"] for r in solv2]
print(paired_counts(a, b), "p =", mcnemar_exact(a, b))
print("Kappa tra le due esecuzioni:", cohen_kappa(a, b))

{'entrambe_giuste': 20, 'solo_a': 0, 'solo_b': 0, 'entrambe_sbagliate': 0} p = 1.0
Kappa tra le due esecuzioni: nan


In [6]:
model0, date0, v0 = load_nihongo(NIHONGO / "results_v0.json")
model1, date1, v1 = load_nihongo(NIHONGO / "results_v1.json")
print(model1, date1, len(v1))
for metric in ["route_ok", "tool_ok", "answer_ok"]:
    print(metric, proportion_summary([r[metric] for r in v1]))

gemini-3.5-flash-lite 2026-11-06 28
route_ok {'k': 25, 'n': 28, 'p': 0.8928571428571429, 'wilson': (0.728, 0.963), 'bootstrap': (0.786, 1.0)}
tool_ok {'k': 26, 'n': 28, 'p': 0.9285714285714286, 'wilson': (0.774, 0.98), 'bootstrap': (0.821, 1.0)}
answer_ok {'k': 25, 'n': 28, 'p': 0.8928571428571429, 'wilson': (0.728, 0.963), 'bootstrap': (0.786, 1.0)}


In [7]:
by_q1 = {r["question"]: r for r in v1}
common = [r for r in v0 if r["question"] in by_q1]
a = [r["tool_ok"] for r in common]
b = [by_q1[r["question"]]["tool_ok"] for r in common]
print(len(common), "domande comuni")
print(paired_counts(a, b), "p =", mcnemar_exact(a, b))
print("Differenza v1 - v0: %.2f  IC95 [%.2f, %.2f]" % paired_bootstrap_diff(a, b))

20 domande comuni
{'entrambe_giuste': 19, 'solo_a': 0, 'solo_b': 1, 'entrambe_sbagliate': 0} p = 1.0
Differenza v1 - v0: 0.05  IC95 [0.00, 0.15]


In [8]:
cost0 = [r["seconds"] for r in common]
cost1 = [by_q1[r["question"]]["seconds"] for r in common]
print("Tempo v1 - v0 (s), media: %.2f  IC95 [%.2f, %.2f]" % paired_bootstrap_diff(cost0, cost1))

Tempo v1 - v0 (s), media: -1.97  IC95 [-3.88, -0.27]


## Lettura dei risultati

1. **Perché per 20/20 riporto Wilson e non il bootstrap.**
Con 20 risposte giuste su 20, ogni ricampionamento bootstrap contiene solo valori 1, quindi l'intervallo collassa a [1,00; 1,00]. Sembra una certezza assoluta, ma è solo un limite del metodo quando la proporzione sta al bordo. Wilson invece tiene conto che 20 domande sono poche e dà 0,84–1,00: con questi dati la vera accuratezza potrebbe anche essere intorno all'85%. Per le proporzioni su campioni piccoli uso Wilson; il bootstrap lo tengo per mediane, medie di token e differenze.

2. **Cosa dice e cosa NON dice il p = 1,0 di v0 contro v1.**
Dice che sulle 20 domande comuni non ho prove di una differenza nella scelta del tool. 19 domande sono giuste con entrambe le versioni, e c'è una sola discordanza (a favore della v1). Una sola coppia discordante non può mai dare un risultato significativo. NON dice che le due versioni siano equivalenti: l'intervallo della differenza va da 0,00 a 0,15, quindi non posso escludere un miglioramento fino a 15 punti. Quello che posso affermare è che la v1 non ha peggiorato nessuna domanda.

3. **Perché la kappa tra le due esecuzioni di nl-sql è nan.**
In entrambe le esecuzioni tutte le 20 query sono corrette, quindi i due "valutatori" usano un'unica etichetta. In questo caso anche l'accordo atteso per caso vale 1, e la formula diventa 0 diviso 0. Non è un errore del codice: la kappa semplicemente non è definita quando non c'è variabilità. Per descrivere la stabilità tra le esecuzioni basta dire che le due esecuzioni coincidono su 20 domande su 20.

4. **La differenza di tempo v1 - v0 esclude lo zero. Posso attribuirla al router?**
No. L'intervallo (da −3,88 a −0,27 secondi) esclude lo zero, ma le due versioni sono state eseguite in giorni diversi, il 30 ottobre e il 6 novembre. La latenza dell'API cambia da un giorno all'altro, quindi l'effetto del progetto è confuso con quello del momento. Anzi, la v1 fa più chiamate al modello (2,61 contro 1,80 per domanda) e quindi dovrebbe essere più lenta, non più veloce. Per attribuire una differenza di tempo al codice dovrei eseguire le due versioni nella stessa sessione, alternando le domande.

5. **Con 20 domande, quante ne devo migliorare (e 0 peggiorare) per p < 0,05 al Giorno B?**
Almeno 6. Con 6 migliorate e 0 peggiorate il test esatto dà p = 0,031; con 5 e 0 dà p = 0,0625, non sufficiente. Contano solo le domande che cambiano esito, quelle che restano uguali non aiutano. Se anche una sola domanda peggiora, ne servono almeno 8 migliorate (p = 0,039). Nel progetto 3 il confronto vero sarà sulle 5 domande impossibili: anche se l'astensione passasse da 0/5 a 5/5, il test darebbe p = 0,0625. Lo riporterò come miglioramento chiaro ma non statisticamente significativo, con gli intervalli di Wilson delle due versioni.